# Foam vs Ice Burn Investigation

Reproducible analysis notebook for the Helios PDD calibration foam-burn deficit investigation.

**Lead question:** Why does Helios's fab007 production run (wetted DT-CH foam main fuel) yield 26 MJ while LILAC's same target yields 87 MJ?

**Companion narrative log:** [`notes/foam_vs_ice_investigation.md`](../notes/foam_vs_ice_investigation.md) — read that first for the storyline. This notebook hosts the reproducible computations.

**To extend:** each section header below has a clear extension point. Add new analyses as new sections following the same pattern (markdown header + code cell + interpretation cell). New runs go in the inventory dict in section 2.

## 1. Setup

Imports and path resolution. Notebook runs from the repo's `notebooks/` directory; `sys.path` insertion lets us import `helios_postprocess` without requiring a `pip install -e .`.

In [ ]:
import sys, os
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path('~/Codes/helios_postprocess').expanduser()
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

print(f'Repo root: {REPO}')
print(f'helios_postprocess on path: {(REPO / "helios_postprocess" / "__init__.py").exists()}')

## 2. Run inventory

All simulation runs analyzed in this investigation. Base paths point to the Studio share mount on the MacBook. To add a new run, append to the dict.

**Headline metrics** come from the per-run `_summary.txt`.

In [ ]:
STUDIO_PDD = Path('/Volumes/tommehlhorn/Sims/Xcimer/Olson_PDD')

RUNS = {
    'fab015-foam'      : STUDIO_PDD / 'Olson_PDD_20_fab015_foot25_s018_c37_burn'      / 'Olson_PDD_20_fab015_foot25_s018_c37_burn',
    'fab007-foam-prod' : STUDIO_PDD / 'Olson_PDD_20_fab007_foot25_s018_c37_burn'      / 'Olson_PDD_20_fab007_foot25_s018_c37_burn',
    'fab007-foam-nb'   : STUDIO_PDD / 'Olson_PDD_20_fab007_foot25_s018_c37_nb'        / 'Olson_PDD_20_fab007_foot25_s018_c37_nb',
    'fab007-ice'       : STUDIO_PDD / 'Olson_PDD_20_fab007_foot25_s018_c37_ice_burn'  / 'Olson_PDD_20_fab007_foot25_s018_c37_ice_burn',
    'fab007-CR'        : STUDIO_PDD / 'Olson_PDD_20_fab007_foot25_s018_c37_CR_burn'   / 'Olson_PDD_20_fab007_foot25_s018_c37_CR_burn',
    'fab02-foam'       : STUDIO_PDD / 'Olson_PDD_20_fab02_foot25_s016_burn'           / 'Olson_PDD_20_fab02_foot25_s016_burn',
}

for label, base in RUNS.items():
    exo = base.with_suffix('.exo')
    print(f'  {label:22s}  exo {"FOUND" if exo.exists() else "missing":7s}  {exo}')

In [ ]:
# Headline metrics table (manually transcribed from the per-run _summary.txt).
# Update if the simulations are rerun.
import pandas as pd

headlines = pd.DataFrame([
    {'run':'fab015-foam',      'V_peak_kms':429, 'coupling_pct':74.4, 'peak_rho_bang':235, 'HS_rhoR_max':0.37, 'unablated_DT_pct':12.6, 'yield_MJ':29.3},
    {'run':'fab007-foam-prod', 'V_peak_kms':421, 'coupling_pct':73.1, 'peak_rho_bang':245, 'HS_rhoR_max':0.35, 'unablated_DT_pct':12.6, 'yield_MJ':26.0},
    {'run':'fab007-ice',       'V_peak_kms':427, 'coupling_pct':73.1, 'peak_rho_bang':310, 'HS_rhoR_max':0.71, 'unablated_DT_pct':45.9, 'yield_MJ':81.4},
    {'run':'fab007-CR',        'V_peak_kms':418, 'coupling_pct':73.1, 'peak_rho_bang':157, 'HS_rhoR_max':0.22, 'unablated_DT_pct':12.9, 'yield_MJ':4.2 },
    {'run':'fab02-foam',       'V_peak_kms':463, 'coupling_pct':84.0, 'peak_rho_bang':None,'HS_rhoR_max':None, 'unablated_DT_pct':None, 'yield_MJ':59.0},
    {'run':'LILAC reference',  'V_peak_kms':410, 'coupling_pct':68.0, 'peak_rho_bang':None,'HS_rhoR_max':0.85, 'unablated_DT_pct':None, 'yield_MJ':87.4},
])
headlines.set_index('run')

## 3. Zero-D rate verification

Reference rate formulas (Bosch-Hale 1992 DT reactivity, NRL bremsstrahlung) used to validate Helios's burn and radiation kernels at a static plasma condition.

**Test protocol:** uniform sphere, R=0.05 cm, 100 zones, hydrodynamics OFF, fixed (ρ, T_e=T_i), compare Helios's reported instantaneous rates vs analytic.

Extension point: add new T or composition points to `verification_results` as runs come back.

In [ ]:
# ---- Bosch-Hale (1992) D-T reactivity, cm^3/s, T in keV ----
def sigmav_DT(T_keV):
    B_G = 34.3827
    mrc2 = 1124656.0
    C1, C2, C3 = 1.17302e-9, 1.51361e-2, 7.51886e-2
    C4, C5     = 4.60643e-3, 1.35000e-2
    C6, C7     = -1.06750e-4, 1.36600e-5
    T = np.asarray(T_keV, dtype=float)
    theta = T / (1.0 - (T*(C2 + T*(C4 + T*C6))) / (1.0 + T*(C3 + T*(C5 + T*C7))))
    xi = (B_G**2 / (4.0*theta))**(1.0/3.0)
    return C1 * theta * np.sqrt(xi/(mrc2 * T**3)) * np.exp(-3.0*xi)

# ---- NRL bremsstrahlung, W/cm^3 ----
def P_brem_NRL(T_keV, n_e_cc, sum_Z2nZ_cc):
    return 1.69e-32 * np.sqrt(T_keV*1000.0) * sum_Z2nZ_cc * n_e_cc

# Physical constants
E_FUS_J = 17.6e6 * 1.602176634e-19   # J per DT reaction
E_ALPHA_J = 3.5e6 * 1.602176634e-19   # alpha share
AMU_G = 1.66053906660e-24             # 1 amu in grams

In [ ]:
# ---- Reference table at n_DT = 1e22 cm^-3 (pure DT, optically thin) ----
n_DT_ref = 1e22
rho_pureDT = n_DT_ref * (2.014 + 3.016)/2 * AMU_G
print(f'Reference density (pure DT, n_DT=1e22 cm^-3): rho = {rho_pureDT*1e3:.3f} mg/cc = {rho_pureDT:.5f} g/cc')

T_grid = np.array([3, 4, 5, 6, 7, 8, 10, 12, 15, 20])
sv = sigmav_DT(T_grid)
R_fus_per_gram = (n_DT_ref**2 / 4.0) * sv / rho_pureDT     # reactions/g/s
P_brem_per_gram = P_brem_NRL(T_grid, n_DT_ref, n_DT_ref) / rho_pureDT  # W/g

ref = pd.DataFrame({
    'T_keV': T_grid,
    'sigmav_cm3_s': sv,
    'R_fus_per_g_s': R_fus_per_gram,
    'P_brem_TW_per_g': P_brem_per_gram * 1e-12,
}).set_index('T_keV')
ref.style.format({'sigmav_cm3_s':'{:.3e}', 'R_fus_per_g_s':'{:.3e}', 'P_brem_TW_per_g':'{:.1f}'})

In [ ]:
# ---- Verification: Helios reported values vs analytic ----
# Each row is a finished test point. Add new rows as runs come back from Studio.
verification_results = pd.DataFrame([
    {'test':'pure-DT @3keV  fusion', 'T_keV':3, 'composition':'pure DT', 'metric':'R_fus #/g/s',  'helios':1.11819e27, 'analytic':1.1177e27},
    {'test':'pure-DT @3keV  brems',  'T_keV':3, 'composition':'pure DT', 'metric':'P_brem TW/g',  'helios':2364.64,    'analytic':2215.0  },
    {'test':'1.7%C  @3keV  fusion',  'T_keV':3, 'composition':'1.7%C CH foam', 'metric':'R_fus #/g/s',  'helios':9.39775e26, 'analytic':9.45e26 },
    {'test':'1.7%C  @3keV  brems',   'T_keV':3, 'composition':'1.7%C CH foam', 'metric':'P_brem TW/g',  'helios':4202.76,    'analytic':3403.0  },
    # ---- to be added ----
    # {'test':'pure-DT @5keV  fusion', 'T_keV':5, ... }
    # {'test':'pure-DT @5keV  brems',  'T_keV':5, ... }
    # ... etc ...
])
verification_results['ratio'] = verification_results['helios'] / verification_results['analytic']
verification_results['verdict'] = verification_results['ratio'].apply(
    lambda r: '✓ within 2%' if abs(r-1)<0.02
              else '✓ within 10% (likely convention/physics)' if abs(r-1)<0.10
              else '✓ within 30% (likely extra physics)' if abs(r-1)<0.30
              else '⚠ investigate')
verification_results.style.format({'helios':'{:.4g}', 'analytic':'{:.4g}', 'ratio':'{:.4f}'})

**Interpretation of zero-D results (as of 2026-05-25):**

- Pure-DT fusion matches Bosch-Hale to 0.04% — reactivity kernel + density bookkeeping are exact
- Pure-DT brems is +6.8% over NRL — consistent with a Gaunt factor ~1.18 vs NRL's averaged 1.11
- 1.7%C foam fusion matches CH-binder analytic to 0.6% — composition handling in burn is correct
- 1.7%C foam brems is +23.5% over NRL — decomposes as ~7% Gaunt convention + ~16% extra physics (Z-dependent Gaunt for carbon + free-bound recombination radiation)

**Net:** basic rate kernels validated. The CR_burn pathology cannot be in the burn or pure-DT brems algorithms. It lives in opacity-table behavior at compressed density, EOS-opacity coupling, or both — the next-priority test is the optical-thickness sweep.

## 4. Foam vs ice production-run comparison

Three figures generated by `examples/compare_runs.py` for the fab015-foam vs fab007-ice pair. Re-running the script regenerates them in-place under `comparisons/`.

Extension point: to add a new comparison pair (e.g. fab007-prod vs CR_burn), invoke `compare_runs.py` with the new labels and add a new section here with the figure embeds.

In [ ]:
COMPARISONS = REPO / 'comparisons'

rt_png        = COMPARISONS / 'compare_foam_vs_ice_rt.png'
histories_png = COMPARISONS / 'compare_foam_vs_ice_histories.png'
lineouts_png  = COMPARISONS / 'compare_foam_vs_ice_lineouts.png'

for p in (rt_png, histories_png, lineouts_png):
    print(f'  {"FOUND" if p.exists() else "MISSING":7s}  {p}')

print('\nTo regenerate (from repo root):')
print('  PYTHONPATH=. python examples/compare_runs.py \\')
print(f'    {RUNS["fab015-foam"]} \\')
print(f'    {RUNS["fab007-ice"]} \\')
print('    --labels foam ice')

In [ ]:
if rt_png.exists():        display(Image(filename=str(rt_png)))
if histories_png.exists(): display(Image(filename=str(histories_png)))
if lineouts_png.exists():  display(Image(filename=str(lineouts_png)))

**Key visual findings (per notes/foam_vs_ice_investigation.md, §3):**

- Drive trajectories essentially identical — comparison is fair
- Central hot-spot ρ: ice 30 g/cc vs foam 12 g/cc
- Central T_ion at bang: **ice 75 keV vs foam 38 keV** — 2x gap
- Ablation eats deeper into foam: foam retains 12.6% of initial DT mass at stagnation vs ice 45.9%
- Foam shows T_ion dip around 170–195 µm — burn front partially arrests through foam-region annulus

## 5. Energy balance ledger

From `examples/energy_balance_diagnostic.py` on the foam-vs-ice pair, stagnation snapshot:

In [ ]:
energy_balance = pd.DataFrame({
    'channel_kJ': ['Absorbed laser', 'Alpha deposited', 'Plasma thermal', 'KE outward (blowoff)', 'Rad escape (boundary)', 'Total fusion released (MJ)'],
    'foam':       [1549.35, 5832.91,  459.63,  6292.27,  670.49, 29.3],
    'ice':        [1491.78, 16185.18, 1922.95, 14475.58, 977.24, 81.4],
})
energy_balance['ratio_ice_over_foam'] = energy_balance['ice'] / energy_balance['foam']
energy_balance.set_index('channel_kJ').style.format('{:.2f}')

**Headline conclusion from the ledger:**

Alpha-deposited / total-fusion-released = **19.9% in both runs** to three digits. Alpha transport is *not* the foam-vs-ice discriminator. The 2.77x yield gap lives entirely upstream in the n²T²σv reaction-rate integral.

PDF report: `{run_A}_vs_{run_B}_energy_balance.pdf` in the repo root (writes to CWD when invoked).

## 6. Ideal ignition vs carbon impurity (Bosch-Hale + NRL)

Verification that carbon brems-cooling and fuel dilution are *not* the foam-burn deficit mechanism. T_ig at 1.7% C is only 5.19 keV — the foam burn region runs at 38–75 keV, far above this threshold.

In [ ]:
from scipy.optimize import brentq

# T_ig such that alpha-heating power density = brems power density (per n_i^2)
# For DT-only contribution from D+T fraction (1-f) of ions:
#   alpha_per_ni2 = ((1-f)/2)^2 * <sv>(T) * E_alpha
#   brems_per_ni2 = 1.69e-32 * sqrt(T_eV) * (1+35f) * (1+5f)
C_B = 1.69e-32 * np.sqrt(1000.0)   # bremsstrahlung prefactor with T in keV
def T_ignite(f):
    def imbal(T):
        lhs = ((1-f)**2/4.0) * sigmav_DT(T) * E_ALPHA_J
        rhs = C_B * (1 + 5*f) * (1 + 35*f) * np.sqrt(T)
        return lhs - rhs
    return brentq(imbal, 1.0, 40.0)

f_grid = np.array([0.0, 0.005, 0.010, 0.017, 0.030, 0.047])
T_ig = np.array([T_ignite(f) for f in f_grid])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(f_grid*100, T_ig, marker='o', color='#c0392b', lw=2)
for f, T in zip(f_grid, T_ig):
    ax.annotate(f'{T:.2f} keV', xy=(f*100, T), xytext=(0, 8),
                textcoords='offset points', ha='center', fontsize=9)
ax.axhspan(38, 75, color='gray', alpha=0.15, label='foam burn-region T_ion (observed)')
ax.set_xlabel('Carbon atomic fraction (%)')
ax.set_ylabel('Ideal ignition temperature T_ig (keV)')
ax.set_title('Ideal ignition T vs C impurity (alpha = brems)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nT_ig (pure DT) = {T_ignite(0.0):.3f} keV')
print(f'T_ig (1.7% C) = {T_ignite(0.017):.3f} keV')
print(f'\n=> foam burn region (38-75 keV) is FAR above T_ig with carbon.')
print(f'   Carbon brems cooling cannot be the dominant suppression mechanism.')

## 7. CR_burn EOS-swap anomaly

`fab007_foot25_s018_c37_CR_burn` has Sesame DT-ice EOS *and* PROPACEOS wetted-foam opacity in the main fuel. Expected ice-like compression with marginal extra rad loss; observed yield collapse to 4.2 MJ — lower than even production foam (26 MJ).

**Three-way comparison (fab007 production foam vs fab007-ice vs CR_burn):**

In [ ]:
cr_comparison = pd.DataFrame([
    {'metric':'Peak rho at bang (g/cc)',  'foam_prod':245.0, 'ice':310.0, 'CR_burn':157.0},
    {'metric':'Adiabat',                  'foam_prod':1.95,  'ice':np.nan,'CR_burn':1.62 },
    {'metric':'HS rhoR peak (g/cm2)',     'foam_prod':0.35,  'ice':0.71,  'CR_burn':0.22 },
    {'metric':'<T_hs> (keV)',             'foam_prod':23.1,  'ice':np.nan,'CR_burn':13.1 },
    {'metric':'Burn width FWHM (ns)',     'foam_prod':0.15,  'ice':0.15,  'CR_burn':0.30 },
    {'metric':'Yield (MJ)',               'foam_prod':26.0,  'ice':81.4,  'CR_burn':4.2  },
])
cr_comparison.set_index('metric')

**Interpretation:** lower adiabat (1.62 vs 1.95) + matched KE delivery should give *more* compression than production foam, not less. Peak ρ collapsed to 157 g/cc (vs production 245). HS ρR never reaches the 0.3 ignition threshold — no bootstrap, marginal burn.

**Three hypotheses for the EOS-swap pathology:**

1. **Opacity-at-compressed-density:** PROPACEOS foam opacity table outside calibrated range at 100s g/cc, over-radiates during implosion
2. **Sesame DT-ice EOS misbehavior with C contamination:** DT-ice EOS phase structure breaks at Z=6 impurity
3. **EOS-opacity inconsistency:** internally different reference compositions break detailed balance

**Next diagnostic:** run `energy_balance_diagnostic.py` on (foam_prod, ice, CR_burn) triplet. If CR_burn rad-escape channel ballooned vs foam_prod → hypothesis (1) confirmed.

## 8. Next planned tests

**Near-term (this week):**

1. Zero-D T sweep, pure DT @ {5, 7, 10, 15} keV — check brems Helios/NRL ratio holds at ~1.07 across T
2. Zero-D optical-thickness sweep at T=10 keV, n_DT ∈ {1e22, 1e23, 1e24, 1e25} — find where rad transport diverges from optically-thin analytic; bears directly on CR_burn opacity hypothesis
3. CR_burn energy-balance diagnostic on (CR_burn, fab007-prod, fab007-ice) triplet

**Medium-term:**

4. Clean Fraley-style microsphere test: uniform sphere, no driver, T_0 ≳ 10 keV; vary material between pure DT, 1.7% C foam, real fab007 foam composition; compare burnup fraction vs Fraley analytic
5. Re-examine FL transition regime (fab003 NO-IGNITION run as the FL knee marker)

**Longer-term:**

6. HDD calibration transfer once foam-burn opacity question is closed (use fab02 settings as anchor, per CLAUDE.md HDD transfer plan)

## 9. Cross-references

- **Narrative log:** [`notes/foam_vs_ice_investigation.md`](../notes/foam_vs_ice_investigation.md)
- **Project memories** (persist across Claude sessions):
  - `~/.claude/projects/-Users-mehlhorn/memory/foam_vs_ice_energy_balance.md`
  - `~/.claude/projects/-Users-mehlhorn/memory/fab007_burn_propagation_diagnostic.md`
- **Project guide:** [`CLAUDE.md`](../CLAUDE.md)
- **Tools used:** [`examples/compare_runs.py`](../examples/compare_runs.py), [`examples/energy_balance_diagnostic.py`](../examples/energy_balance_diagnostic.py), [`examples/dump_burn_propagation_profile.py`](../examples/dump_burn_propagation_profile.py)